# Visual Odometry Workshop
Welcome to the VO Workshop!

In this workshop, we're going to learn how to use feature tracking to build an odometry estimation algorithm! This is very useful in Visual SLAM systems, often considered Step #1.
<P>
So let's begin! We will do this in 3 steps:
1. Feature Tracking (Detection, Description, Matching)
2. Pose Recovery (E & F, R & T)
3. Visual Odometry Graph

But first, let's do some imports...

## **Waymo Open Dataset & Imports**

In [ ]:
!wget -qq https://optical-flow-data.s3.eu-west-3.amazonaws.com/waymo_images.zip
!unzip -qq waymo_images.zip && rm waymo_images.zip
!mkdir output
!ls

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pickle
from google.colab.patches import cv2_imshow

In [ ]:
#TODO: Load Different Image Pairs
img1 = cv2.imread("downtown/front_images_downtown/1557197711848851.jpg")
img2 = cv2.imread("downtown/front_images_downtown/1557197711948687.jpg")

def to_rgb(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

plt.imshow(to_rgb(img2))

The sample images will be used to experiment with the feature tracking, but frankly, in Visual SLAM, we need a complete sequence.

In [ ]:
import json

video_path = 'downtown/front_camera_downtown.mp4'
cap = cv2.VideoCapture(video_path)

# Define the path to the camera calibration JSON file
calibration_json_path = 'downtown/camera_calibration_downtown.json'

print("Video file path: ", video_path)
print("Calibration JSON path: ", calibration_json_path)

# Open and read the JSON file
with open(calibration_json_path, 'r') as f:
    calibration_data = json.load(f)

# Extract camera intrinsics matrix and distortion coefficients
camera_matrix = np.array(calibration_data['intrinsics'])
dist_coeffs = np.array(calibration_data['distortion'])

print("Camera Intrinsics Matrix (K):")
print(camera_matrix)
print("\nDistortion Coefficients:")
print(dist_coeffs)

## **vSLAM Process**

In [ ]:
# INITIALIZE FAST DETECTOR AND BRIEF DESCRIPTOR
fast = cv2.xfeatures2d.StarDetector_create()
brief = cv2.xfeatures2d.BriefDescriptorExtractor_create()

FLANN_INDEX_KDTREE = 0
index_params = dict(algorithm = FLANN_INDEX_KDTREE, trees = 5)
search_params = dict(checks = 50)

flann = cv2.FlannBasedMatcher(index_params, search_params)

print("Images loaded and feature objects initialized.")

In [ ]:
# DETECTOR
kp1 = fast.detect(img1,None)

# DESCRIPTOR
kp1, des1 = brief.compute(img1, kp1)

# DETECTOR
kp2 = fast.detect(img2,None)
# DESCRIPTOR
kp2, des2 = brief.compute(img2, kp2)

In [ ]:
img1_kp = cv2.drawKeypoints(img1, kp1, None, color=(0,255,0), flags=0)
img2_kp = cv2.drawKeypoints(img2, kp2, None, color=(0,255,0), flags=0)

fig = plt.figure(figsize=(15, 7))
ax1 = plt.subplot(121)
ax2 = plt.subplot(122)

ax1.imshow(to_rgb(img1_kp))
ax2.imshow(to_rgb(img2_kp))
plt.show()

In [ ]:
#FLANN MATCHING
FLANN_INDEX_KDTREE = 0
index_params = dict(algorithm = FLANN_INDEX_KDTREE, trees = 5)
search_params = dict(checks = 50)

flann = cv2.FlannBasedMatcher(index_params, search_params)

matches = flann.knnMatch(np.float32(des1),np.float32(des2),k=2) # Use NP.FLOAT32 for ORB, BRIEF, etc

# store all the good matches as per Lowe's ratio test.
good = []
for m,n in matches:
    if m.distance < 0.7*n.distance:
        good.append(m)

if len(good)>10:
    p1 = np.float32([ kp1[m.queryIdx].pt for m in good ]).reshape(-1,1,2)
    p2 = np.float32([ kp2[m.trainIdx].pt for m in good ]).reshape(-1,1,2)

draw_params = dict(matchColor = (0,255,0), # draw matches in green color
                    singlePointColor = None,
                    flags = 2)

img_briefmatch = cv2.drawMatches(img1,kp1,img2,kp2,good,None,**draw_params)
cv2_imshow(img_briefmatch)

In [ ]:
draw_params = dict(matchColor = (0,255,0), # draw matches in green color
                    singlePointColor = None,
                    flags = 2)


In [ ]:
matches = flann.knnMatch(np.float32(des1), np.float32(des2), k=2)

good = []
for m, n in matches:
    if m.distance < 0.7 * n.distance:
        good.append(m)

print(f"Found {len(good)} good matches after Lowe's ratio test.")

In [ ]:
img_briefmatch = cv2.drawMatches(img1,kp1,img2,kp2,good,None,**draw_params)
cv2_imshow(img_briefmatch)

In [ ]:
p1 = np.float32([ kp1[m.queryIdx].pt for m in good ]).reshape(-1,1,2)
p2 = np.float32([ kp2[m.trainIdx].pt for m in good ]).reshape(-1,1,2)

print(f"Extracted {len(p1)} points from img1 (p1) and {len(p2)} points from img2 (p2).")
print("Camera matrix and distortion coefficients are ready for use.")

In [ ]:
p1_undistorted = cv2.undistortPoints(p1, camera_matrix, dist_coeffs, P=camera_matrix)
p2_undistorted = cv2.undistortPoints(p2, camera_matrix, dist_coeffs, P=camera_matrix)

# Reshape to 2D array for cv2.findEssentialMat
p1_undistorted_flat = p1_undistorted.reshape(-1, 2)
p2_undistorted_flat = p2_undistorted.reshape(-1, 2)

# Estimate Essential Matrix
E, mask_E = cv2.findEssentialMat(p1_undistorted_flat, p2_undistorted_flat, camera_matrix, method=cv2.RANSAC, prob=0.999, threshold=1.0)

# Recover Pose
_, R_rel, t_rel, mask_pose = cv2.recoverPose(E, p1_undistorted_flat, p2_undistorted_flat, camera_matrix, mask=mask_E)

print("Matched points undistorted.")
print("Essential Matrix estimated.")
print("Relative Rotation (R_rel) and Translation (t_rel) recovered.")
print("R_rel:\n", R_rel)
print("t_rel:\n", t_rel)

In [ ]:
import matplotlib.pyplot as plt

# Define the initial camera position
initial_camera_pos = (0, 0)

# Extract X and Z components from t_rel
# t_rel is a 3x1 vector: [[X], [Y], [Z]]
# For a top-down view, we are interested in X (lateral) and Z (forward) movement.
# The coordinate system assumes Z is forward, X is right, and Y is down.
# So, t_rel[0] is X, t_rel[2] is Z.
second_camera_x = t_rel[0, 0]
second_camera_z = t_rel[2, 0]

# Create a new Matplotlib figure
plt.figure(figsize=(8, 8))

# Plot the initial camera position
plt.plot(initial_camera_pos[0], initial_camera_pos[1], 'ro', markersize=10, label='Camera 1 Position (Origin)')

# Plot the second camera's position
plt.plot(second_camera_x, second_camera_z, 'go', markersize=10, label='Camera 2 Position')

# Draw a dashed blue line connecting the two positions
plt.plot([initial_camera_pos[0], second_camera_x], [initial_camera_pos[1], second_camera_z], 'b--', label='Relative Translation')

# Set labels and title
plt.xlabel('X-coordinate (Lateral Movement)')
plt.ylabel('Z-coordinate (Forward Movement)')
plt.title('2D Plot of Camera Positions for Two Frames (Top-Down View)')

# Add a grid
plt.grid(True)

# Ensure equal scaling for the X and Z axes
plt.axis('equal')

# Display the legend
plt.legend()

# Show the plot
plt.show()

In [ ]:
trajectory_points = []
trajectory_points.append(t.flatten())

### **Exercise: Add a 3rd Camera?**

In [ ]:
img3 = cv2.imread("downtown/front_images_downtown/1557197712048522.jpg")
# DETECTOR
kp3 = #TODO
# DESCRIPTOR
kp3, des3 = #TODO
img3_kp = #TODO

plt.imshow(to_rgb(img3_kp))
plt.show()

In [ ]:
### MATCH
matches = #TODO

good = []
for m, n in matches:
    if m.distance < 0.7 * n.distance:
        #TODO

print(f"Found {len(good)} good matches after Lowe's ratio test.")
img_briefmatch23 = #TODO
cv2_imshow(img_briefmatch23)

In [ ]:
# The 'good' matches here refer to the matches between des2 and des3 from the previous cell OilWm56R-YYj

# Extract points from kp2 using queryIdx from the good matches (des2 -> des3)
p2_for_img2img3 = #TODO
p3_for_img2img3 = #TODO

# Undistort points for the current match
p2_undistorted_for_img2img3 = #TODO
p3_undistorted_for_img2img3 = #TODO

# Reshape to 2D array for cv2.findEssentialMat
p2_undistorted_flat_for_img2img3 = #TODO
p3_undistorted_flat_for_img2img3 = #TODO

# Estimate Essential Matrix
E, mask_E = #TODO
# Recover Pose
_, R_rel_23, t_rel_23, mask_pose = #TODO

print("Recovered Relative Rotation Matrix (R_rel_23):\n", R_rel_23)
print("Recovered Relative Translation Vector (t_rel_23):\n", t_rel_23)


In [ ]:
# Define the initial camera position
initial_camera_pos = (0, 0)

# Extract X and Z components from t_rel (from img1 to img2) and flip Z for plotting convention
# Assuming Z is depth/forward, and we want positive values to mean forward/up in the plot.
# If t_rel's Z component is typically negative for forward motion, we multiply by -1.
second_camera_x = t_rel[0, 0]
second_camera_z = -t_rel[2, 0] # Flip sign for plotting: positive Z means moving forward/up

# Extract X and Z components from t_rel_23 (from img2 to img3) and flip Z for plotting convention
# We will also flip the Z component of t_rel_23 for consistent plotting.
effective_t_rel_23_x = t_rel_23[0,0]
effective_t_rel_23_z = -t_rel_23[2,0] # Flip sign for plotting: positive Z means moving forward/up

# Cumulative position of Camera 3 relative to Camera 1
absolute_third_camera_x = second_camera_x + effective_t_rel_23_x
absolute_third_camera_z = second_camera_z + effective_t_rel_23_z

# Create a new Matplotlib figure
plt.figure(figsize=(8, 8))

# Plot the initial camera position (Camera 1)
plt.plot(initial_camera_pos[0], initial_camera_pos[1], 'ro', markersize=10, label='Camera 1 Position (Origin)')

# Plot the second camera's position (Camera 2)
plt.plot(second_camera_x, second_camera_z, 'go', markersize=10, label='Camera 2 Position')

# Plot the third camera's position (Camera 3)
plt.plot(absolute_third_camera_x, absolute_third_camera_z, 'bo', markersize=10, label='Camera 3 Position')

# Draw dashed lines connecting the positions to form a trajectory
plt.plot([initial_camera_pos[0], second_camera_x, absolute_third_camera_x],
         [initial_camera_pos[1], second_camera_z, absolute_third_camera_z], 'k--', label='Trajectory Path')

# Set labels and title
plt.xlabel('X-coordinate (Lateral Movement)')
plt.ylabel('Z-coordinate (Forward Movement)')
plt.title('2D Plot of Accumulated Camera Positions (Top-Down View)')

# Add a grid
plt.grid(True)

# Ensure equal scaling for the X and Z axes
plt.axis('equal')

# Display the legend
plt.legend()

# Show the plot
plt.show()


---

### **Solution**

In [ ]:
img3 = cv2.imread("downtown/front_images_downtown/1557197712048522.jpg")
# DETECTOR
kp3 = fast.detect(img3,None)
# DESCRIPTOR
kp3, des3 = brief.compute(img3, kp3)
img3_kp = cv2.drawKeypoints(img3, kp3, None, color=(0,255,0), flags=0)

plt.imshow(to_rgb(img3_kp))
plt.show()

In [ ]:
### MATCH
matches = flann.knnMatch(np.float32(des2), np.float32(des3), k=2)

good = []
for m, n in matches:
    if m.distance < 0.7 * n.distance:
        good.append(m)

print(f"Found {len(good)} good matches after Lowe's ratio test.")
img_briefmatch23 = cv2.drawMatches(img2,kp2,img3,kp3,good,None,**draw_params)
cv2_imshow(img_briefmatch23)

In [ ]:
# The 'good' matches here refer to the matches between des2 and des3 from the previous cell OilWm56R-YYj

# Extract points from kp2 using queryIdx from the good matches (des2 -> des3)
p2_for_img2img3 = np.float32([ kp2[m.queryIdx].pt for m in good ]).reshape(-1,1,2)
p3_for_img2img3 = np.float32([ kp3[m.trainIdx].pt for m in good ]).reshape(-1,1,2)

# Undistort points for the current match
p2_undistorted_for_img2img3 = cv2.undistortPoints(p2_for_img2img3, camera_matrix, dist_coeffs, P=camera_matrix)
p3_undistorted_for_img2img3 = cv2.undistortPoints(p3_for_img2img3, camera_matrix, dist_coeffs, P=camera_matrix)

# Reshape to 2D array for cv2.findEssentialMat
p2_undistorted_flat_for_img2img3 = p2_undistorted_for_img2img3.reshape(-1, 2)
p3_undistorted_flat_for_img2img3 = p3_undistorted_for_img2img3.reshape(-1, 2)

# Estimate Essential Matrix
E, mask_E = cv2.findEssentialMat(p2_undistorted_flat_for_img2img3, p3_undistorted_flat_for_img2img3, camera_matrix, method=cv2.RANSAC, prob=0.999, threshold=1.0)

# Recover Pose
_, R_rel_23, t_rel_23, mask_pose = cv2.recoverPose(E, p2_undistorted_flat_for_img2img3, p3_undistorted_flat_for_img2img3, camera_matrix, mask=mask_E)

print("Recovered Relative Rotation Matrix (R_rel_23):\n", R_rel_23)
print("Recovered Relative Translation Vector (t_rel_23):\n", t_rel_23)


In [ ]:
# Define the initial camera position
initial_camera_pos = (0, 0)

# Extract X and Z components from t_rel (from img1 to img2) and flip Z for plotting convention
# Assuming Z is depth/forward, and we want positive values to mean forward/up in the plot.
# If t_rel's Z component is typically negative for forward motion, we multiply by -1.
second_camera_x = t_rel[0, 0]
second_camera_z = -t_rel[2, 0] # Flip sign for plotting: positive Z means moving forward/up

# Extract X and Z components from t_rel_23 (from img2 to img3) and flip Z for plotting convention
# We will also flip the Z component of t_rel_23 for consistent plotting.
effective_t_rel_23_x = t_rel_23[0,0]
effective_t_rel_23_z = -t_rel_23[2,0] # Flip sign for plotting: positive Z means moving forward/up

# Cumulative position of Camera 3 relative to Camera 1
absolute_third_camera_x = second_camera_x + effective_t_rel_23_x
absolute_third_camera_z = second_camera_z + effective_t_rel_23_z

# Create a new Matplotlib figure
plt.figure(figsize=(8, 8))

# Plot the initial camera position (Camera 1)
plt.plot(initial_camera_pos[0], initial_camera_pos[1], 'ro', markersize=10, label='Camera 1 Position (Origin)')

# Plot the second camera's position (Camera 2)
plt.plot(second_camera_x, second_camera_z, 'go', markersize=10, label='Camera 2 Position')

# Plot the third camera's position (Camera 3)
plt.plot(absolute_third_camera_x, absolute_third_camera_z, 'bo', markersize=10, label='Camera 3 Position')

# Draw dashed lines connecting the positions to form a trajectory
plt.plot([initial_camera_pos[0], second_camera_x, absolute_third_camera_x],
         [initial_camera_pos[1], second_camera_z, absolute_third_camera_z], 'k--', label='Trajectory Path')

# Set labels and title
plt.xlabel('X-coordinate (Lateral Movement)')
plt.ylabel('Z-coordinate (Forward Movement)')
plt.title('2D Plot of Accumulated Camera Positions (Top-Down View)')

# Add a grid
plt.grid(True)

# Ensure equal scaling for the X and Z axes
plt.axis('equal')

# Display the legend
plt.legend()

# Show the plot
plt.show()
